# 07b · Weather: ERA5 single levels from ARCO-ERA5

The same surface weather as `era5_single` in notebook 07 (boundary-layer height, cloud base, CAPE, CIN, K index, heat fluxes, sunshine, cloud cover, temperature, dew point, wind at 10 and 100 m, gusts, pressure, rain), read from **ARCO-ERA5**, Google's public copy of ERA5 on Cloud Storage. No account, no key and no queue: it reads one global field per variable and hour, many at a time.

For each site it keeps its competition days and the day before each, from 5 to 21 h local solar time. Each month is saved to `database/weather/arco/single/` as soon as it is read, so **if Colab disconnects, Run All again** and it carries on where it stopped. Hours after ARCO's last date are left to the CDS (notebook 07).

The last steps build `era5_single.parquet` and compare it with the pilots' thermals.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
THREADS = 32         # global fields read at a time
EVERY = 1            # read every hour
SOLAR = "5-21"       # local solar hours kept for each site
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, get the code

In [ ]:
import pandas as pd
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "site_chips.parquet")), "Run notebook 06 first: database/site_chips.parquet is missing"
subprocess.run(["pip", "install", "-q", "gcsfs", "zarr", "netCDF4"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script (otherwise it would
    keep running in the background)."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        return proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")

## 3. What there is to read

How many hours and global fields, and whether some hours are newer than ARCO (those come from the CDS). Nothing is read yet.

In [ ]:
run(f"{REPO}/scripts/era5_arco.py", DB, "--product", "single", "--threads", str(THREADS), "--every", str(EVERY), "--solar", SOLAR, "--plan-only")

## 4. Read

One line per month, with its speed and the time left. Stopping this cell also stops the reading; months already saved are kept.

In [ ]:
run(f"{REPO}/scripts/era5_arco.py", DB, "--product", "single", "--threads", str(THREADS), "--every", str(EVERY), "--solar", SOLAR)

## 5. Build the table

Merges the months into `weather/era5_single.parquet` with any files already downloaded from the CDS.

In [ ]:
run(f"{REPO}/scripts/era5.py", DB, "--convert-only", "--tables", "single")

## 6. First check: ERA5 boundary layer vs real thermal tops

In [ ]:
import duckdb
db = duckdb.connect()
W = os.path.join(DB, "weather")
db.execute(f"CREATE VIEW era5_single AS SELECT * FROM read_parquet('{W}/era5_single.parquet')")
db.execute(f"CREATE VIEW climbs AS SELECT * FROM read_parquet('{DB}/climbs.parquet')")
db.execute(f"CREATE VIEW tasks AS SELECT * FROM read_parquet('{DB}/tasks.parquet')")

# First check against the pilots: ERA5 boundary-layer height vs the tops of real thermals, same chip and hour
import sys; sys.path.insert(0, REPO)
from pta import grid
c = db.sql("""
    SELECT c.task_key, c.lat, c.lon, c.alt_top, c.t_start, k.date
    FROM climbs c JOIN tasks k USING (task_key)
    WHERE c.is_thermal AND k.date IS NOT NULL
    USING SAMPLE 200000
""").df()
c["chip_id"] = grid.chip_id(c["lat"].to_numpy(), c["lon"].to_numpy())
c["time"] = (pd.to_datetime(c["date"]) + pd.to_timedelta(c["t_start"], unit="s")).dt.round("h")
w = db.sql("SELECT chip_id, time, blh_m FROM era5_single").df()
m = c.merge(w, on=["chip_id", "time"])
# thermal top above the ground (the chip's mean elevation from cells)
cells = pd.read_parquet(f"{DB}/cells.parquet", columns=["chip_id", "elev_mean"]).groupby("chip_id").elev_mean.mean()
m["ground"] = m["chip_id"].map(cells)
m["top_agl"] = m["alt_top"] - m["ground"]
print(f"{len(m):,} thermals matched to ERA5")
print(m[["top_agl", "blh_m"]].describe().round(0))
print("correlation:", round(m["top_agl"].corr(m["blh_m"]), 2))